**RAG Pipeline:**

Document ───▶ Tokens ───▶ Embeddings ───▶ Vector Database ───▶ Ask User Query (Q) ───▶ Retrieve Top-k Docs ───▶ Context Assembly ───▶ LLM(Generator) ───▶ Final Answer

1. Install Library

In [ ]:
!pip install langchain==0.3.27
!pip install langchain-core==0.3.72
!pip install langchain-community==0.3.27
# Document file format - PDF
!pip install pypdf
# Document file format - Tiktoken
!pip install tiktoken
# vector database
!pip install chromadb
# vector database - faiss [others - Weaviate, Pinecone]
!pip install faiss-cpu
# Install google generative ai
!pip install langchain-google-genai==2.0.9
# Install google generative ai
!pip install google-generativeai==0.7.2

  Using cached langchain_core-0.3.86-py3-none-any.whl.metadata (3.2 kB)
Using cached langchain_core-0.3.86-py3-none-any.whl (461 kB)
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 0.3.72
    Uninstalling langchain-core-0.3.72:
      Successfully uninstalled langchain-core-0.3.72
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langgraph 1.2.6 requires langchain-core<2,>=1.4.7, but you have langchain-core 0.3.86 which is incompatible.
langgraph-sdk 0.4.2 requires langchain-core<2,>=1.4.0, but you have langchain-core 0.3.86 which is incompatible.
langgraph-prebuilt 1.1.0 requires langchain-core>=1.3.1, but you have langchain-core 0.3.86 which is incompatible.
  Using cached langchain_core-0.3.72-py3-none-any.whl.metadata (5.8 kB)
Using cached langchain_core-0.3.72-py3-none-any.whl (442 kB)
  Attempting uninstall: langchain-c

2. Import Library

In [ ]:
from langchain.document_loaders import PyPDFLoader, DirectoryLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain.chains import RetrievalQA

3. Loading PDF

In [ ]:
loader = DirectoryLoader('/content/drive/MyDrive/Agentic AI - 29-06/Tesla', glob='**/*.pdf',
                         loader_cls=PyPDFLoader)

In [ ]:
documents = loader.load()

In [ ]:
print(f"Number of documents loaded from directory : {len(documents)}")

Number of documents loaded from directory : 664


4. Recursive Character TextSplitter

In [ ]:
# Recursive Splitter using "\n\n" (Paragraph), "\n" Newline, " "(space), ".", ""(Empty String), ";"
# (semi-color) - chunking using maximum size as 500.
recursive_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100,
                                                    separators = ["\n\n","\n"," ","",".",";",","])

recursive_tokens = recursive_splitter.split_documents(documents)

In [ ]:
print(f"Number of chunks created from documents: {len(recursive_tokens)}")

Number of chunks created from documents: 5694


5. Embeddings

In [ ]:
# Pre-trained embedding model available in huggingface
hf_embeddings = HuggingFaceEmbeddings(model_name = "sentence-transformers/all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

6. VectorStore (FAISS)

In [ ]:
from langchain_community.vectorstores import FAISS

In [ ]:
# faiss_store = FAISS.from_documents(recursive_tokens, hf_embeddings)

In [ ]:
# faiss_store.save_local("/content/drive/MyDrive/Agentic AI - 29-06/faiss_index")

In [ ]:
faiss_store = FAISS.load_local("/content/drive/MyDrive/Agentic AI - 29-06/faiss_index", hf_embeddings,
                               allow_dangerous_deserialization=True)

7. LLM Model - Gemini

In [ ]:
import os
os.environ["GOOGLE_API_KEY"] = "AQ.Ab8RN6LXmiqvcGZlT7HJSy4pgaeAgLl62WT0a67NDh0nAzTdLA"

In [ ]:
import google.generativeai as genai
genai.configure(api_key = "AQ.Ab8RN6LXmiqvcGZlT7HJSy4pgaeAgLl62WT0a67NDh0nAzTdLA")

In [ ]:
for model in genai.list_models():
    print(model.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/gemini-3-pro-image
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-3.5-flash
models/gemini-omni-flash-preview
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/gemini-3.1-flash-tts-preview
models/gemini-robotics-er-1.5-preview
mod

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

In [ ]:
llm = ChatGoogleGenerativeAI(
    model = "gemini-flash-lite-latest",
    temperature = 0.7,
    max_tokens=2048
)

8. Create a Buffer Memory to Store History of Conversation

In [ ]:
from langchain.memory import ConversationBufferMemory

In [ ]:
# ConversationBufferMemory stores entire conversation history It does not have built-in limit on number of conversation or messages
memory = ConversationBufferMemory(memory_key = "chat_history",
                                  return_messages = True,
                                  output_key="answer")

In [ ]:
# similarity : rank chunks purely by cosine distance between the query and the chunk embeddings.
# search_type : "similarity" the top-k most similar chunks are returned closest to the query vector.
# max marginal relevance (mmr): Rank chunks by a combination of their similarity to the query
# and their diversity relative to the already selected chunks. This does a two step process, it
# first feteches a larger candidate pool then iteratively selects the most relevant and diverse
# chunks until it has selected "k" chunks.
retriever = faiss_store.as_retriever(search_type = "mmr", search_kwargs = {"k": 3})

9. Create ChatPromptTemplate

In [ ]:
from langchain_core.prompts import PromptTemplate

In [ ]:
prompt = PromptTemplate(
    input_variables=["context","chat_history","question"],
    template = """You are a financial analyst with extensive experience in analysing Tesla Financial Reports.
    Use only Tesla's financial document as source of context to provide the response to each query or question. Always mention the fiscal
    year and quarter in your answer if the information is not available be concise and accurate in your response. If the Answer is not
    Available in the provided documents say "I dont have knowledge."

    Context: {context}
    Chat History: {chat_history}
    Question: {question}
    Answer:""")

10. Create ConversationalRetrievalChain using LLM and Memory

In [ ]:
from langchain.chains import ConversationalRetrievalChain

In [ ]:
QA_Chain = ConversationalRetrievalChain.from_llm(
    llm = llm,
    retriever = retriever,
    memory = memory,
    return_source_documents = True,
    verbose = False,
    combine_docs_chain_kwargs = {"prompt": prompt},
    output_key = "answer"
)

print("Welcome to the Tesla Financial Report Analysis ChatBot!")

Welcome to the Tesla Financial Report Analysis ChatBot!


11. RAG Query

In [ ]:
def ask_question(question: str) -> dict:
    result = QA_Chain.invoke({"question": question})
    print("="*70)
    print(f"Q: {question}")
    print(f"A: {result['answer']}")
    print("-"*70)
    print("Sources")
    for i, doc in enumerate(result["source_documents"], start = 1):
        meta = doc.metadata
        source = meta.get("source", "Unknown").split("\\")[-1]     # filename only
        page   = meta.get("page", "?")
        print(f"  [{i}] {source}  |  Page {page}")
        print(f"       {doc.page_content[:150].strip()}...")
    print("=" * 70)

In [ ]:
ask_question("What was Tesla's total revenue in fiscal year 2023?")

Q: What was Tesla's total revenue in fiscal year 2023?
A: Tesla's total revenue for the fiscal year ended December 31, 2023, was $96,773 million.
----------------------------------------------------------------------
Sources
  [1] /content/drive/MyDrive/Agentic AI - 29-06/Tesla/tsla-20231231-gen.pdf  |  Page 50
       Tesla,	Inc.
Consolidated	Statements	of	Operations
(in	millions,	except	per	share	data)
Year	Ended	December	31,
2023
2022
2021
Revenues
Automotive	sale...
  [2] /content/drive/MyDrive/Agentic AI - 29-06/Tesla/tsla-20231231-gen.pdf  |  Page 88
       2.23
	billion	of	annual	tax	revenues	starting	at	the	end	of	2023.	As	of	December	31,
2023,	we	have	met	and	expect	to	meet	the	tax	revenue	requirements...
  [3] /content/drive/MyDrive/Agentic AI - 29-06/Tesla/tsla-20251231-gen.pdf  |  Page 40
       Services	and	other	revenue	increased	$2.00	billion,	or	19%,	in	the	year	ended	December	31,	2025	as	compared	to	the	year	ended	December	31,
2024,	prima...


In [ ]:
ask_question("What are different types of Tesla Car ?")

Q: What are different types of Tesla Car ?
A: Based on the provided document, Tesla currently manufactures five different consumer vehicle models: the Model 3, Model Y, Model S, Model X, and the Cybertruck. (Note: The provided text does not specify a fiscal year or quarter for this information).
----------------------------------------------------------------------
Sources
  [1] /content/drive/MyDrive/Agentic AI - 29-06/Tesla/tsla-20251231-gen.pdf  |  Page 7
       more	affordable,	maintain	and	strengthen	the	Tesla	brand	and	obtain	rapid	customer	feedback.
We	reevaluate	our	sales	strategy	both	globally	and	at	a	l...
  [2] /content/drive/MyDrive/Agentic AI - 29-06/Tesla/tsla-20241231-gen.pdf  |  Page 4
       Our	Products	and	Services
Automotive
We	currently	manufacture	five	different	consumer	vehicles	–	the	Model	3,	Y,	S,	X	and	Cybertruck.	Model	3	is	a	fou...
  [3] /content/drive/MyDrive/Agentic AI - 29-06/Tesla/tsla-20221231-gen.pdf  |  Page 4
       traction	and	performance	in	an	all